# <i class="fas fa-star"></i>&hairsp;T08: Crossover Detection

## Introduction

When tracing neurons in dense neuropil, neurites frequently pass close to each other without actually connecting—these are **crossovers**. Detecting crossovers is important for:

- **Quality control**: Identifying potential tracing errors where crossovers were mistakenly connected
- **Topology validation**: Verifying that apparent intersections in 2D projections are truly separate in 3D
- **Morphological analysis**: Understanding spatial relationships between neurites

SNT's `CrossoverFinder` utility detects locations where two distinct paths approach within a distance threshold but do not share a topological connection (i.e., no branch point at that location).

### Overview

In this tutorial, we'll demonstrate crossover detection in pySNT using sample neuronal reconstructions. We will cover:

1. **Basic crossover detection** with default settings
2. **Advanced configuration** for fine-tuning detection parameters
3. **Visualization** in both BigVolumeViewer (BVV) and napari
4. **Exporting** crossover locations for downstream analysis

```{admonition} Learning Objectives
:class: tip

By the end of this tutorial, you'll understand how to detect, visualize, and export crossover events from neuronal reconstructions.

**Estimated Time**: 15–20 minutes
```

```{note}
Make sure to read these resources before running this notebook:
- [Install](../install.md) - Installation instructions
- [Tutorial 01](./01_single_cell_analysis.ipynb) - Working with Trees
- [Tutorial 05](./05_napari_viewer.ipynb) - Visualization basics
```

## Setup and Initialization

We start by initializing pySNT in interactive mode to enable 3D visualization:

In [ ]:
import pysnt
pysnt.set_option('java.logging.level', 'Error')
pysnt.set_option('display.chart_format', 'svg')
pysnt.initialize(mode='interactive')

## Loading Sample Data

We'll use the OP_1 dataset from the [DIADEM Challenge](https://diadem.janelia.org/), which contains olfactory projection neurons with several crossover locations. This dataset is ideal because:

- It contains multiple neurites in close proximity
- The tracing includes several near-miss locations
- It's small enough for quick demonstration

In [ ]:
from pysnt import SNTService, Tree

snt_service = SNTService()
tree = snt_service.demoTree('OP1')
print(f"Loaded: {tree.getLabel()}")
print(f"Number of paths: {tree.list().size()}")
print(f"Number of nodes: {tree.getNodesAsPoint().size()}")

Let's visualize the tree to get a sense of its structure:

In [ ]:
pysnt.display(tree)

## Basic Crossover Detection

The `CrossoverFinder` utility detects spatial locations where paths approach each other closely but don't share a topological connection. Let's start with the default configuration:

In [ ]:
from pysnt.util import CrossoverFinder

# Get paths from the tree
paths = tree.list()

# Create default configuration
config = CrossoverFinder.Config()
print(f"Default config: {config}")

# Find crossovers
events = CrossoverFinder.find(paths, config)
print(f"\nFound {len(events)} crossover event(s)")

Each `CrossoverEvent` contains:
- `x`, `y`, `z`: Coordinates of the crossover center (in calibrated units)
- `participants`: The paths involved in the crossover
- `medianAngleDeg`: The crossing angle between local tangents (0–90°)
- `medianMinDist`: The closest approach distance between segments

Let's examine the detected events:

In [ ]:
for i, event in enumerate(events):
    print(f"\nEvent {i + 1}:")
    print(f"  Location: ({event.x:.2f}, {event.y:.2f}, {event.z:.2f})")
    print(f"  Crossing angle: {event.medianAngleDeg:.1f}°")
    print(f"  Min distance: {event.medianMinDist:.3f}")
    print(f"  Participants: {event.participants.size()} path(s)")

## Understanding CrossoverFinder Configuration

The `CrossoverFinder.Config` class provides fine-grained control over detection. Here are the key parameters:

| Parameter | Default | Description |
|-----------|---------|-------------|
| `proximity` | 2.0 | Neighborhood radius in calibrated units (e.g., µm) |
| `thetaMinDeg` | 0.0 | Minimum crossing angle (0 disables filtering) |
| `minRunNodes` | 2 | Minimum consecutive near node-pairs required |
| `sameCTOnly` | true | Only compare paths with same channel/time |
| `includeSelfCrossovers` | false | Detect crossovers within the same path |
| `includeDirectChildren` | false | Include parent-child crossovers |
| `nodeWitnessRadius` | -1 | Post-filter radius (falls back to proximity if ≤0) |

Let's explore how these parameters affect detection:

In [ ]:
# Stricter configuration: larger proximity, angle filter
strict_config = CrossoverFinder.Config() \
    .proximity(5.0) \
    .thetaMinDeg(30) \
    .minRunNodes(3)

strict_events = CrossoverFinder.find(paths, strict_config)
print(f"With stricter config: {len(strict_events)} crossover(s)")

# More permissive: smaller proximity, no angle filter
permissive_config = CrossoverFinder.Config() \
    .proximity(1.0) \
    .thetaMinDeg(0) \
    .minRunNodes(1)

permissive_events = CrossoverFinder.find(paths, permissive_config)
print(f"With permissive config: {len(permissive_events)} crossover(s)")

### Self-Crossovers

By default, crossovers within the same path (self-crossovers) are ignored. These can occur in highly tortuous neurites. Enable detection with `includeSelfCrossovers(true)`:

In [ ]:
self_config = CrossoverFinder.Config() \
    .proximity(3.0) \
    .includeSelfCrossovers(True)

self_events = CrossoverFinder.find(paths, self_config)
print(f"Including self-crossovers: {len(self_events)} event(s)")

## Visualization in BigVolumeViewer (BVV)

Let's visualize the crossover locations in 3D using BVV. We'll mark each crossover with a sphere:

In [ ]:
from pysnt.viewer import Bvv
from pysnt.util import SNTColor

# Create BVV instance
bvv = Bvv()

# Add the tree
bvv.add(tree)

# Show the viewer. An image is required
img = snt_service.demoImage('OP1')
bvv.show(img)

if events:
    # Visualize crossover events locations
    for event in events:
        # Create annotation points
        bvv.getMarkerManager().add(event.x, event.y, event.z, SNTColor.fromString('red'), 5) # coords, color, radius
else:
    print("No crossovers to display")

## Visualization in napari

We can also visualize crossovers in napari, which provides interactive 2D/3D viewing:

In [ ]:
import napari
import numpy as np

# Create napari viewer
viewer = napari.Viewer(ndisplay=3)

# Get tree skeleton as points for napari
tree_points = []
for path in paths:
    for i in range(path.size()):
        node = path.getNode(i)
        tree_points.append([node.z, node.y, node.x])  # napari uses ZYX order

tree_array = np.array(tree_points)

# Add tree points
viewer.add_points(
    tree_array,
    name='Neuron',
    size=1,
    face_color='white',
    opacity=0.5
)

# Add crossover markers
if events:
    crossover_array = np.array([[e.z, e.y, e.x] for e in events])
    viewer.add_points(
        crossover_array,
        name='Crossovers',
        size=5,
        face_color='red',
        opacity=1.0
    )
    print(f"Added {len(events)} crossover markers to napari")

## Exporting Crossover Locations

Crossover locations can be exported for use in other tools or further analysis. Here we demonstrate exporting to CSV and as ImageJ ROIs:

In [ ]:
import csv
import os

# Export to CSV
output_dir = os.path.expanduser('~')
csv_path = os.path.join(output_dir, 'crossovers.csv')

with open(csv_path, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['ID', 'X', 'Y', 'Z', 'Angle_deg', 'MinDist', 'NumParticipants'])
    for i, event in enumerate(events):
        writer.writerow([
            i + 1,
            f"{event.x:.3f}",
            f"{event.y:.3f}",
            f"{event.z:.3f}",
            f"{event.medianAngleDeg:.1f}",
            f"{event.medianMinDist:.3f}",
            event.participants.size()
        ])

print(f"Exported {len(events)} crossovers to: {csv_path}")

### Export as ImageJ Point ROIs

For integration with ImageJ workflows, we can export crossovers as point ROIs:

In [ ]:
from scyjava import jimport

# Import ImageJ classes
PointRoi = jimport('ij.gui.PointRoi')
RoiManager = jimport('ij.plugin.frame.RoiManager')

# Get or create ROI Manager
rm = RoiManager.getRoiManager()
if rm is None:
    rm = RoiManager()

# Add each crossover as a point ROI
for i, event in enumerate(events):
    # Get XYZCT coordinates (ImagePlus convention)
    xyzct = event.xyzct()
    x, y, z = xyzct[0], xyzct[1], xyzct[2]
    
    # Create point ROI
    roi = PointRoi(x, y)
    roi.setName(f"Crossover_{i + 1}")
    roi.setPosition(int(z) + 1)  # 1-based slice index
    rm.addRoi(roi)

print(f"Added {len(events)} point ROIs to ROI Manager")

## Working with Multiple Trees

CrossoverFinder can also detect crossovers between paths from different trees. This is useful when analyzing multiple neurons that may interact spatially:

In [ ]:
# Load another demo tree
tree2 = snt_service.demoTree('pyramidal')

# Combine paths from both trees
from java.util import ArrayList
combined_paths = ArrayList()
for path in tree.list():
    combined_paths.add(path)
for path in tree2.list():
    combined_paths.add(path)

# Find crossovers across both trees
# Note: This may produce many results if trees don't share coordinate space
multi_config = CrossoverFinder.Config().proximity(10.0)
multi_events = CrossoverFinder.find(combined_paths, multi_config)
print(f"Cross-tree analysis: {len(multi_events)} potential crossover(s)")

## Practical Tips

### Choosing the Right Proximity Threshold

The `proximity` parameter should be set based on your data's spatial calibration and expected neurite diameter. A good starting point is 2–3× the typical neurite radius.

### Using Angle Filtering

The `thetaMinDeg` parameter filters out near-parallel approaches. Set this to:
- **0°**: Detect all crossovers (default)
- **30–45°**: Filter out near-parallel paths
- **60–90°**: Only detect near-perpendicular crossings

### Performance Considerations

CrossoverFinder uses a spatial hash grid for efficient neighbor queries. For very large datasets with many paths, consider:
- Reducing `proximity` to limit candidate pairs
- Increasing `minRunNodes` to filter noise
- Processing subsets of paths if memory is constrained

In [ ]:
# Clean up
# pysnt.dispose()

## Summary

This tutorial introduced SNT's crossover detection capabilities:

**Key Concepts:**
- **Crossovers** are spatial locations where paths approach closely without topological connection
- `CrossoverFinder.Config` provides fine-grained control over detection parameters
- Detection is based on proximity thresholds, crossing angles, and run lengths

**Workflow Pattern:**
1. Load reconstruction(s) and extract paths
2. Configure detection parameters based on data characteristics
3. Run `CrossoverFinder.find()` to detect events
4. Visualize in BVV or napari for validation
5. Export locations for downstream analysis

**Use Cases:**
- Quality control of manual tracings
- Validation of automated reconstruction
- Spatial relationship analysis between neurites

## Follow-up Questions

1. How would you modify the configuration to detect only high-angle crossings (near-perpendicular)?

2. For time-lapse data with multiple timepoints, how would you use the `sameCTOnly` parameter?

## References

- Brown KM, Barrionuevo G, Canty AJ, et al. The DIADEM data sets: representative light microscopy images of neuronal morphology to advance automation of digital reconstructions. Neuroinformatics. 2011;9(2-3):143-157. doi:[10.1007/s12021-010-9095-5](https://doi.org/10.1007/s12021-010-9095-5)

See [SNT citation](https://imagej.net/plugins/snt/faq#how-do-i-cite-snt) for details on how to properly cite SNT.